# Notebook Runner

## Imports and LLM client solving ambiguous cases

In [0]:
from openai import OpenAI
import os

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://adb-6748159052539178.18.azuredatabricks.net/serving-endpoints"
)

In [0]:
import re
import json
import base64
import os
import time
from datetime import datetime
from types import SimpleNamespace
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, ObjectType
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, DoubleType, LongType
)

w = WorkspaceClient()


catalog = dbutils.widgets.get("catalog")
schemas_raw = dbutils.widgets.get("schemas")
switch_config_path = dbutils.widgets.get("switch_config_path")
foundation_model = dbutils.widgets.get("foundation_model")
registry_schema = dbutils.widgets.get("registry_schema")
volume_root = f"/Volumes/{catalog}"

schemas = [s.strip() for s in schemas_raw.split(",") if s.strip()]

## Create Registry Table Mapping Tables to their Catalogs and Schemas

In [0]:
# ── Regex pour extraire le nom de l'objet créé ───────────────────────────────
OBJECT_PATTERN = re.compile(
    r"""
    CREATE\s+(?:OR\s+ALTER\s+)?
    (?P<obj_type>
        TABLE|VIEW|
        (?:INLINE\s+TABLE-VALUED\s+)?FUNCTION|
        (?:STORED\s+)?PROC(?:EDURE)?
    )\s+
    (?:\[?(?P<schema_prefix>\w+)\]?\.)?\s*
    \[?(?P<obj_name>\w+)\]?
    """,
    re.IGNORECASE | re.VERBOSE,
)

def detect_object_type(raw_type: str) -> str:
    t = raw_type.upper()
    if "VIEW"     in t: return "VIEW"
    if "FUNCTION" in t: return "FUNCTION"
    if "PROC"     in t: return "PROCEDURE"
    return "TABLE"

def read_sql_file(path: str) -> str:
    """Read a .sql file, trying UTF-16 (common for SSMS exports) then UTF-8."""
    # Check for BOM to pick the right encoding on the first try
    with open(path, "rb") as f:
        raw = f.read(4)

    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        encodings = ["utf-16"]
    else:
        encodings = ["utf-8-sig", "utf-8", "utf-16", "latin-1"]

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, UnicodeError):
            continue

    # Last resort: read as binary and decode lossily
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

# ── Scan ─────────────────────────────────────────────────────────────────────
rows = []

for schema in schemas:
    level3 = os.path.join(volume_root, schema, f"{schema}_volume", schema)

    if not os.path.isdir(level3):
        print(f"[SKIP] {level3} introuvable")
        continue

    sql_files = [f for f in sorted(os.listdir(level3)) if f.lower().endswith(".sql")]
    print(f"[{schema}] {len(sql_files)} fichiers .sql trouvés")

    for filename in sql_files:
        file_path = os.path.join(level3, filename)

        try:
            content = read_sql_file(file_path)
        except Exception as e:
            print(f"  [WARN] Impossible de lire {file_path}: {e}")
            continue

        match = OBJECT_PATTERN.search(content)
        if match:
            obj_name        = match.group("obj_name")
            obj_type        = detect_object_type(match.group("obj_type"))
            declared_schema = match.group("schema_prefix") or schema
        else:
            parts           = os.path.splitext(filename)[0].split(".")
            obj_name        = parts[1] if len(parts) >= 2 else parts[0]
            obj_type        = "UNKNOWN"
            declared_schema = schema

        rows.append(Row(
            catalog         = catalog,
            schema          = schema,
            declared_schema = declared_schema,
            object_name     = obj_name.lower(),
            object_type     = obj_type,
            file_path       = file_path,
            file_name       = filename,
        ))

print(f"\nTotal : {len(rows)} objets détectés.")

# ── Écriture Delta ────────────────────────────────────────────────────────────
df = spark.createDataFrame(rows)
df.display()

target_table = f"{catalog}.{registry_schema}.sql_object_registry"

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Table écrite : {target_table}")

## Create Collision Table containing Number of Occurences of a Table

In [0]:
from databricks.sdk.service.workspace import ImportFormat, ExportFormat, Language

# ── Regex ─────────────────────────────────────────────────────────────────────
REF_PATTERN = re.compile(
    r'`?\{catalog\}`?\.`?\{schema\}`?\.`?(\w+)`?',
    re.IGNORECASE
)

# ── Registry ──────────────────────────────────────────────────────────────────
registry_df = spark.table("dbe_dbx_internships.log.sql_object_registry")

counts = (
    registry_df.groupBy("object_name")
    .agg(
        F.count("*").alias("cnt"),
        F.collect_set("declared_schema").alias("schemas"),
        F.first("declared_schema").alias("single_schema"),
    )
)

simple_rows     = counts.filter("cnt = 1").select("object_name", "single_schema").collect()
registry_simple = {row.object_name.lower(): f"{catalog}.{row.single_schema}" for row in simple_rows}

collision_rows       = counts.filter("cnt > 1").select("object_name", "schemas").collect()
collision_candidates = {row.object_name.lower(): row.schemas for row in collision_rows}

# ── File paths for collision candidates ───────────────────────────────────────
collision_file_rows = (
    registry_df
    .filter(F.col("object_name").isin(list(collision_candidates.keys())))
    .select("object_name", "declared_schema", "file_path")
    .collect()
)


print(f"Registry : {len(registry_simple)} non-ambigu, {len(collision_candidates)} en collision")

## Call LLM

In [0]:
import yaml

with open(switch_config_path, "r") as f:
    cfg = yaml.safe_load(f)

# ── LLM ─────────────────────────────────────────────────────────────────────────
SYSTEM_PROMPT    = cfg["llm"]["system_prompt"]

CONTENT_EXCERPT  = cfg["llm"]["content_excerpt_size"]
BUSINESS_CONTEXT = cfg["business_context"]

# ── Pairs ─────────────────────────────────────────────────────────────────────
POSSIBLE_PAIRS = cfg["possible_pairs"]
PAIR_KEYS      = [f"{p['catalog']}.{p['schema']}" for p in POSSIBLE_PAIRS]


def call_llm(content: str, ambiguous: dict, source_sql: str = "") -> dict:
    pairs_str     = "\n".join(f"{i+1}) {k}" for i, k in enumerate(PAIR_KEYS))
    ambiguous_str = json.dumps(ambiguous, indent=2)

    user_prompt = f"""\
        # PySpark/SQL query (notebook excerpt):
        {content[:CONTENT_EXCERPT]}

        # Business context:
        {BUSINESS_CONTEXT.strip()}

        # Ambiguous objects to resolve:
        {ambiguous_str}

        # Source SQL file that generated this notebook:
        {source_sql[:CONTENT_EXCERPT]}

        # Possible catalog.schema pairs:
        {pairs_str}

        For each ambiguous object, pick the most likely pair based on the query, business context, and source file definitions.
        Return ONLY a JSON object matching the provided schema.
        """

    response_format = {
      "type": "json_schema",
      "json_schema": {
        "name": "ambiguous_replacement",
        "schema": {
          "type": "object",
          "properties": {
            "table": { "type": "string" },
            "catalog": { "type": "string" },
            "schema": { "type": "string" },
            }
          },
        }
      }
    

    response = client.chat.completions.create(
        model=foundation_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=response_format,

    )

    text = response.choices[0].message.content
    return json.loads(text)


## Resolve Ambiguous and Unambiguous Cases

In [0]:
def resolve_unambiguous(content: str) -> str:
    def replace_ref(match):
        obj_name = match.group(1).lower()
        if obj_name in registry_simple:
            resolved = registry_simple[obj_name]  # already "catalog.schema"
            has_backticks = match.group(0).endswith('`')
            name_part = f'`{match.group(1)}`' if has_backticks else match.group(1)
            return f"{resolved}.{name_part}"
        return match.group(0)

    return REF_PATTERN.sub(replace_ref, content)
  
# ── Résolution ambiguë ────────────────────────────────────────────────────────
def resolve_ambiguous(content_patched: str, ambiguous: dict, source_sql: str = "") -> dict:
    if not ambiguous:
        return {}
    
    result = call_llm(content_patched, ambiguous, source_sql)
    return result


## Process notebook

In [0]:
# ── Orchestration ─────────────────────────────────────────────────────────────
def process_notebook(obj) -> str:

    # 1. Export
    try:
        export_resp = w.workspace.export(obj.path, format=ExportFormat.SOURCE)
        content     = base64.b64decode(export_resp.content).decode("utf-8")
    except Exception as e:
        print(f"  [ERROR] Export {obj.path}: {e}")
        return None

    # 2. Detect refs
    refs_found = {m.group(1).lower() for m in REF_PATTERN.finditer(content)}
    if not refs_found:
        return None

    ambiguous = {name: collision_candidates[name] for name in refs_found if name in collision_candidates}

    # 3. Unambiguous resolution
    content_patched = resolve_unambiguous(content)

    # 4. Ambiguous resolution (LLM)
    if ambiguous:
        print(f"  [LLM] {len(ambiguous)} collision(s): {list(ambiguous.keys())}")

        row = spark.table("dbe_dbx_internships.log.run_details") \
            .filter(F.col("notebook_path") == obj.path) \
            .select("input_file_path") \
            .first()

        source_sql = ""
        if row:
            try:
                with open(row.input_file_path, "r") as f:
                    source_sql = f.read()
            except Exception as e:
                source_sql = f"[ERROR reading source: {e}]"

        llm_resolutions  = resolve_ambiguous(content_patched, ambiguous, source_sql)
        resolved_catalog = llm_resolutions.get("catalog", "")
        resolved_schema  = llm_resolutions.get("schema", "")
        resolved_table   = llm_resolutions.get("table", "").lower()

        def replace_ambiguous(match):
            obj_name = match.group(1).lower()
            if obj_name == resolved_table and resolved_catalog and resolved_schema:
                has_backticks = match.group(0).endswith("`")
                name_part = f"`{match.group(1)}`" if has_backticks else match.group(1)
                return f"{resolved_catalog}.{resolved_schema}.{name_part}"
            return match.group(0)

        content_patched = REF_PATTERN.sub(replace_ambiguous, content_patched)

    # 5. Skip if nothing changed
    if content_patched == content:
        return None

  # 6. Delete then import — but only delete if import-to-temp succeeds first
    temp_path = obj.path + "_tmp_patch"
# Step A: validate by importing to temp path
    w.workspace.import_(
        temp_path,
        content  = base64.b64encode(content_patched.encode()).decode(),
        format   = ImportFormat.SOURCE,
        language = Language.PYTHON,          # ← required for SOURCE + single file
    )

    # Step B: delete original
    w.workspace.delete(obj.path, recursive=True)
    w.workspace.delete(temp_path, recursive=True)

    # Step C: import to correct path
    w.workspace.import_(
        obj.path,
        content  = base64.b64encode(content_patched.encode()).decode(),
        format   = ImportFormat.SOURCE,
        language = Language.PYTHON,
    )

## Replace placeholders

In [0]:
result_schema = StructType([
    StructField("input_file_number", LongType()),
    StructField("input_file_path", StringType()),
    StructField("input_file_relative_path", StringType()),
    StructField("notebook_path", StringType()),
    StructField("run_status", StringType()),
    StructField("run_error", StringType()),
    StructField("run_timestamp", TimestampType()),
    StructField("run_duration_seconds", DoubleType()),
])

for schema in schemas:

    switch_table = spark.sql(f"""
    SELECT table_name
    FROM {catalog}.information_schema.tables
    WHERE table_schema = '{schema}'
      AND table_name LIKE 'lakebridge_switch_%'
    ORDER BY table_name DESC
    LIMIT 1
    """).collect()[0]["table_name"]

    full_switch_table = f"{catalog}.{schema}.{switch_table}"
    # --- Drive from the Switch results table ---
    rows = spark.sql(f"""
        SELECT input_file_number, input_file_path, input_file_relative_path, export_output_path
        FROM {full_switch_table}
    """).collect()
    print(rows)
    output_folder = f"/Workspace/Users/depoplimontj@delawareconsulting.com/delaware_lakebridge/switch_runs/{schema}"

    def _resolve_notebook_path(relative_path, schema, output_folder):
        """Strip the leading schema prefix from relative_path to avoid duplication."""
        prefix = f"{schema}/"
        if relative_path.startswith(prefix):
            relative_path = relative_path[len(prefix):]
        return os.path.join(output_folder, os.path.splitext(relative_path)[0])

    def _object_type_sort_key(row):
        """Sort key: Tables first (0), then other objects (1), then Views last (2)."""
        path = row["input_file_relative_path"]
        if ".Table." in path:
            return (0, row["input_file_number"])
        elif ".View." in path:
            return (2, row["input_file_number"])
        else:
            return (1, row["input_file_number"])

    # --- Step 1: Replace placeholders ---
    for row in rows:
        notebook_path = row["export_output_path"]
        if not notebook_path:
            print(f"  [SKIP] Row {row['input_file_number']} — no export_output_path")
            continue
        process_notebook(SimpleNamespace(path=notebook_path))

## Notebook runs

In [0]:
    # --- Step 2: Run and track (Tables first, then Views) ---
    run_results = []
    sorted_rows = sorted(rows, key=_object_type_sort_key)

    for row in sorted_rows:
        notebook_path = _resolve_notebook_path(
            row["input_file_relative_path"], schema, output_folder
        )
        start = time.time()
        run_ts = datetime.now()

        try:
            dbutils.notebook.run(notebook_path, timeout_seconds=100000, arguments={
                "catalog": catalog,
                "schema": schema
            })
            run_results.append(Row(
                input_file_number=row["input_file_number"],
                input_file_path=row["input_file_path"],
                input_file_relative_path=row["input_file_relative_path"],
                notebook_path=notebook_path,
                run_status="SUCCESS",
                run_error=None,
                run_timestamp=run_ts,
                run_duration_seconds=round(time.time() - start, 2)
            ))
        except Exception as e:
            error_msg = str(e)
            status = "TIMEOUT" if "TimeoutException" in error_msg else "FAILED"
            run_results.append(Row(
                input_file_number=row["input_file_number"],
                input_file_path=row["input_file_path"],
                input_file_relative_path=row["input_file_relative_path"],
                notebook_path=notebook_path,
                run_status=status,
                run_error=error_msg[:2000],
                run_timestamp=run_ts,
                run_duration_seconds=round(time.time() - start, 2)
            ))
            print(f"[ERROR] {notebook_path}: {error_msg[:300]}")

    # --- Step 3: Write tracking results ---
    if run_results:
        spark.createDataFrame(run_results, schema=result_schema).write.mode("append").saveAsTable(
            f"{catalog}.log.run_details"
        )
        print(f"[{schema}] {sum(1 for r in run_results if r.run_status == 'SUCCESS')}/{len(run_results)} notebooks succeeded")
